# Hands On Lab — Train / Validation / Test Splits
**Week 4 — Day 1 | BinX Tech AI & ML Internship**

---

## Objective

In the learning notebook I understood why a three-way split is needed
and what role each dataset plays.

In this lab I apply the full workflow on a different dataset:

1. Create a correct 60/20/20 train/validation/test split
2. Compare three models using the validation set only
3. Tune the best model's hyperparameter using the validation set only
4. Evaluate the final model on the test set exactly once
5. Explain what would have gone wrong if I had tuned against the test set

**Dataset:** Student Performance (2,392 students, 5-class grade prediction)  

---

## Step 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score

SEED = 42

## Step 2 — Load & Inspect the Data

The Student Performance dataset contains 2,392 students with features
like study time, absences, parental support, and extracurricular activities.

The target is `GradeClass` — a 5-class label (0 = A through 4 = F).
This is a multiclass classification problem, so I'll use
**weighted F1** as the metric throughout — it accounts for the
fact that class sizes are very unequal.

I drop `StudentID` because it's just a row identifier, not a feature.

In [2]:
df = pd.read_csv("../../Week1/mini_project/Data/Student_performance_data _.csv")
df = df.drop(columns=["StudentID"])

print("Shape:", df.shape)
print("\nFirst 3 rows:")
print(df.head(3))
print("\nMissing values:", df.isnull().sum().sum())

Shape: (2392, 14)

First 3 rows:
   Age  Gender  Ethnicity  ParentalEducation  StudyTimeWeekly  Absences  \
0   17       1          0                  2        19.833723         7   
1   18       0          0                  1        15.408756         0   
2   15       0          2                  3         4.210570        26   

   Tutoring  ParentalSupport  Extracurricular  Sports  Music  Volunteering  \
0         1                2                0       0      1             0   
1         0                1                0       0      0             0   
2         0                2                0       0      0             0   

        GPA  GradeClass  
0  2.929196         2.0  
1  3.042915         1.0  
2  0.112602         4.0  

Missing values: 0


## Step 3 — Inspect the Target Variable

Before splitting I want to understand the class distribution.
An imbalanced target affects how I interpret scores and why
`stratify` is important here.

In [3]:
# Check how many students belong to each grade class
print("GradeClass distribution:")
print(df["GradeClass"].value_counts().sort_index())

print()

# Check the proportion of each class
print("Class proportions:")
print(
    df["GradeClass"]
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
)

GradeClass distribution:
GradeClass
0.0     107
1.0     269
2.0     391
3.0     414
4.0    1211
Name: count, dtype: int64

Class proportions:
GradeClass
0.0    0.045
1.0    0.112
2.0    0.163
3.0    0.173
4.0    0.506
Name: proportion, dtype: float64


Class 4 (grade F) dominates at 50.6% of the data. This imbalance
is exactly why I need stratify — without it, one split might
accidentally get far more F-grade students than another, making
comparisons between splits unreliable.

## Step 4 — Create the Three-Way Split

Two calls to `train_test_split`:
- First: carve out 20% as the locked test set
- Second: divide the remaining 80% into 75% training / 25% validation

Result: **60% train / 20% validation / 20% test**

`stratify` in both calls keeps the class proportions consistent.

In [4]:
# Separate features (X) from the target (y)
X = df.drop(columns=["GradeClass"])
y = df["GradeClass"].astype(int)


# Step 1: Set aside 20% for the final test set
# The test set is locked immediately and won't be
# used during model selection or hyperparameter tuning.

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

# Step 2: Split the remaining 80%
#          into 60% training and 20% validation
# 75% of the remaining 80% → Training = 60% overall
# 25% of the remaining 80% → Validation = 20% overall

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=SEED,
    stratify=y_temp
)

# Check the final split sizes

total_rows = len(X)

print("Dataset split:")
print(f"Training:   {len(X_train)} rows ({len(X_train) / total_rows:.0%})")
print(f"Validation: {len(X_val)} rows ({len(X_val) / total_rows:.0%})")
print(f"Test:       {len(X_test)} rows ({len(X_test) / total_rows:.0%}) ")

print(f"\nTotal rows: {len(X_train) + len(X_val) + len(X_test)}")

Dataset split:
Training:   1434 rows (60%)
Validation: 479 rows (20%)
Test:       479 rows (20%) 

Total rows: 2392


## Step 5 — Verify Class Balance

With 5 unequal classes, I need to confirm that `stratify` kept
the proportions consistent. I'll check class 4 (the dominant one)
across all three splits.

In [5]:
# Check the class proportions in each dataset split
for name, subset in [
    ("Full dataset", y),
    ("Training", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    proportions = subset.value_counts(normalize=True).sort_index()

    print(f"{name}:")

    for class_id in range(5):
        proportion = proportions.get(class_id, 0)
        print(f"  Class {class_id}: {proportion:.2f}")

    print()

Full dataset:
  Class 0: 0.04
  Class 1: 0.11
  Class 2: 0.16
  Class 3: 0.17
  Class 4: 0.51

Training:
  Class 0: 0.04
  Class 1: 0.11
  Class 2: 0.16
  Class 3: 0.17
  Class 4: 0.51

Validation:
  Class 0: 0.05
  Class 1: 0.11
  Class 2: 0.16
  Class 3: 0.17
  Class 4: 0.51

Test:
  Class 0: 0.04
  Class 1: 0.11
  Class 2: 0.16
  Class 3: 0.17
  Class 4: 0.51



The proportions are nearly identical across all three splits.
stratify is doing its job — each split is a fair representative
sample of the full dataset.

## Step 6 — Compare Three Models on Validation

I'll train three models and compare their **validation F1 only**.
The test set stays untouched.

The three candidates:
- **Logistic Regression** — simple linear baseline (needs scaling)
- **Decision Tree** — rule-based, no scaling needed
- **Random Forest** — ensemble of trees, no scaling needed

I use `make_pipeline` for Logistic Regression so the scaler is
fit on training data only — no leakage into validation.

In [6]:

# Define the models we want to compare

models = {

    # Logistic Regression needs feature scaling,
    # so we put the scaler and model inside one pipeline.
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            random_state=SEED
        )
    ),

    # Decision Tree does not require feature scaling.
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=SEED
    ),

    # Random Forest is an ensemble of multiple decision trees.
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=SEED
    )
}

# Store the results of each model


model_results = {}

# Train and evaluate each model

for name, model in models.items():

    # Train the model using the training set only.
    model.fit(X_train, y_train)

    # Evaluate on training data.
    # This helps us see whether the model is overfitting.
    # average="weighted" because GradeClass has 5 unequal classes.
    # Binary F1 (no average) only works for 2-class problems.
    train_predictions = model.predict(X_train)
    train_f1 = f1_score(
        y_train,
        train_predictions,
        average="weighted"
    )

    # Evaluate on validation data.
    # This score is used to compare the models.
    val_predictions = model.predict(X_val)
    # Same weighted average for consistency
    val_f1 = f1_score(
        y_val,
        val_predictions,
        average="weighted"
    )

    # Save the model and its scores.
    model_results[name] = {
        "model": model,
        "train_f1": train_f1,
        "val_f1": val_f1
    }

# Display the comparison

print("Model Comparison")
print("-" * 45)
print(f"{'Model':<25} {'Train F1':>10} {'Val F1':>10}")
print("-" * 45)

for name, results in model_results.items():

    print(
        f"{name:<25} "
        f"{results['train_f1']:>10.3f} "
        f"{results['val_f1']:>10.3f}"
    )

Model Comparison
---------------------------------------------
Model                       Train F1     Val F1
---------------------------------------------
Logistic Regression            0.805      0.796
Decision Tree                  0.942      0.898
Random Forest                  1.000      0.904


**Reading the results:**

Random Forest wins on validation F1 (0.904) so it becomes the model
I'll tune in the next step.

Two things worth noting here:
- Logistic Regression has the smallest train/val gap (0.009) — it's
  the least prone to overfitting, but it's also the least powerful
  on this dataset.
- Random Forest scores a perfect 1.000 on training — it memorized
  every training example. But its validation F1 (0.904) is still
  the highest, meaning the ensemble's averaging reduces overfitting
  enough to generalize well.

The decision is made on **validation F1**, not training F1.

## Step 7 — Tune the Best Model on Validation

Random Forest won the model comparison. Now I tune its `max_depth`
using the validation set — still keeping the test set locked.

In [7]:
tune_results = {}

# Try different max_depth values
depth_values = [3, 5, 10, None]

for depth in depth_values:

    # Create and train a Random Forest
    model = RandomForestClassifier(
        max_depth=depth,
        n_estimators=100,
        random_state=SEED
    )

    model.fit(X_train, y_train)

    # F1 on training data
    train_predictions = model.predict(X_train)
    train_f1 = f1_score(
        y_train,
        train_predictions,
        average="weighted"
    )

    # F1 on validation data
    # This is the score used to choose the best depth
    val_predictions = model.predict(X_val)
    val_f1 = f1_score(
        y_val,
        val_predictions,
        average="weighted"
    )

    # Store the results
    tune_results[depth] = {
        "train_f1": train_f1,
        "val_f1": val_f1
    }


# Display the results
print("Random Forest Hyperparameter Tuning")
print("-" * 45)
print(f"{'max_depth':<15} {'Train F1':>10} {'Val F1':>10}")
print("-" * 45)

for depth, results in tune_results.items():
    print(
        f"{str(depth):<15}"
        f"{results['train_f1']:>10.3f}"
        f"{results['val_f1']:>10.3f}"
    )


# Choose the depth with the highest validation F1
best_depth = max(
    tune_results,
    key=lambda depth: tune_results[depth]["val_f1"]
)

print(f"\nBest max_depth: {best_depth}")
print(
    f"Best Validation F1: "
    f"{tune_results[best_depth]['val_f1']:.3f}"
)

Random Forest Hyperparameter Tuning
---------------------------------------------
max_depth         Train F1     Val F1
---------------------------------------------
3                   0.844     0.803
5                   0.890     0.864
10                  0.975     0.901
None                1.000     0.904

Best max_depth: None
Best Validation F1: 0.904


**Reading the results:**

Unlike the Pima dataset in the learning notebook (where `max_depth=3`
won), here `max_depth=None` gives the best validation F1.

This makes sense — with 2,392 samples and 13 features, there's enough
data for the trees to grow deep without memorizing noise. The ensemble
averaging across 100 trees controls variance even when individual trees
are fully grown.

This is a practical reminder: there's no single "best" hyperparameter
that works everywhere. The right value depends on the dataset.

## Step 8 — Final Evaluation on the Test Set 

All decisions are made:
- Model chosen: Random Forest
- max_depth chosen: None (by validation F1)

I now open the test set for the **first and only time**.

In [8]:
# Create the final Random Forest using the best max_depth
final_model = RandomForestClassifier(
    max_depth=best_depth,
    n_estimators=100,
    random_state=SEED
)

# Train the final model on the training set
final_model.fit(X_train, y_train)

# Make predictions on the test set
test_predictions = final_model.predict(X_test)

# Calculate final performance on the test set
test_f1 = f1_score(
    y_test,
    test_predictions,
    average="weighted"
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

# Show the validation score used for selecting the model
print(f"Validation F1: {tune_results[best_depth]['val_f1']:.3f}")

# Show the final test performance
print(f"Test F1: {test_f1:.3f}")
print(f"Test Accuracy: {test_accuracy:.3f}")

Validation F1: 0.904
Test F1: 0.910
Test Accuracy: 0.916


The test F1 (0.910) is actually slightly higher than the validation F1
(0.904) — this is normal and can happen with larger datasets where both
splits are stable representatives of the full distribution. The key
point is that 0.910 is trustworthy because the test set had no influence
on any decision I made.

## Step 9 — What Would Have Gone Wrong?

Suppose I had tuned `max_depth` against the test set instead of validation:

```text
max_depth=3    → Test F1 = 0.808
max_depth=5    → Test F1 = 0.867
max_depth=10   → Test F1 = 0.907
max_depth=None → Test F1 = 0.910  ← I would have chosen this
```

Every time I checked the test set, I gave the model indirect information
about those 479 specific rows. By the end, my chosen configuration reflects
what works best on that particular test slice — not what generalizes best
to truly new data.

The "final" test score would no longer be an honest estimate.
It would measure how well I optimized for 479 specific rows,
not how the model performs in the real world.

The discipline is simple: the test set answers one question only —
**"How does my finalized model perform on data it's never seen?"**
The moment it answers a development question, its value as an honest estimator is gone.

---

## Summary

| Step | What I Did | Key Decision |
|------|-----------|--------------|
| Split | 60/20/20 with stratify | Test locked immediately |
| Model selection | Compared 3 models on val F1 | Random Forest (0.904) |
| Hyperparameter tuning | Tried 4 depths on val F1 | max_depth=None (0.904) |
| Final evaluation | Test set opened once | Test F1 = 0.910  |

**The rule that made this evaluation trustworthy:**
the test set had zero influence on any decision until the very end.

The slight gap between val F1 and test F1 is normal —
different data slices give slightly different scores.
What matters is that the test score is honest.

**Next:** Cross-validation replaces the single validation set with
k rotating folds, giving a more stable estimate during development.